# Train v22 on Workstation

This notebook extracts the packaged code locally and launches v22 training in-process so logs appear directly in the notebook. Keep API keys in environment variables or a local `.env`; do not paste secrets into cells.

In [ ]:
import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

VERSION = 'v22'
DRIVE_CODE_DIR = Path('G:/My Drive/quant-research-workbench/workstation_code') / VERSION
PACKAGE_ZIP = DRIVE_CODE_DIR / f'inhouse_transformer_{VERSION}_workstation.zip'
MANIFEST_PATH = DRIVE_CODE_DIR / 'workstation_manifest.json'
LOCAL_CODE_ROOT = Path(f'D:/TradingCodes/quant-research-workbench-{VERSION}-runtime')

assert PACKAGE_ZIP.exists(), f'Missing package: {PACKAGE_ZIP}'
assert MANIFEST_PATH.exists(), f'Missing manifest: {MANIFEST_PATH}'
manifest = json.loads(MANIFEST_PATH.read_text())
print(json.dumps(manifest, indent=2))

if LOCAL_CODE_ROOT.exists():
    shutil.rmtree(LOCAL_CODE_ROOT)
LOCAL_CODE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as package:
    package.extractall(LOCAL_CODE_ROOT)
sys.path.insert(0, str(LOCAL_CODE_ROOT))
print('installed code at', LOCAL_CODE_ROOT)


In [ ]:
# Edit FLATFILES_ROOT if you copy data from the HDD/Drive path to local SSD/NVMe.
FLATFILES_ROOT = Path(manifest['default_flatfiles_root'])
CACHE_ROOT = Path(manifest['default_cache_root'])
OUTPUT_ROOT = Path(manifest['default_output_root'])
print('flatfiles root:', FLATFILES_ROOT, 'exists=', FLATFILES_ROOT.exists())
print('cache root:', CACHE_ROOT)
print('output root:', OUTPUT_ROOT)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
%pip install -q polars pyarrow wandb torchinfo torchview graphviz


Optional: run the profiling cell before choosing chunk/cap settings. This is available for v22 and skipped for older versions.

In [ ]:
import subprocess
import sys

RUN_PROFILE = False  # Set True to profile chunk sizes and event caps before preprocessing.
PROFILE_PROCESSES = int(manifest.get('default_preprocess_processes', 8))
POLARS_THREADS_PER_PROCESS = int(manifest.get('default_polars_threads_per_process', 2))
profile_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'profile_event_chunks.py'
profile_args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--start-date', manifest['train_start_date'],
    '--end-date', manifest['validation_end_date'],
    '--tickers', manifest.get('tickers', 'ALL'),
    '--chunk-ms', '100,250,500,1000',
    '--caps', '64,128,256,512',
    '--processes', str(PROFILE_PROCESSES),
    '--polars-threads-per-process', str(POLARS_THREADS_PER_PROCESS),
]
if RUN_PROFILE:
    print('Running:', ' '.join([str(profile_py), *profile_args]))
    subprocess.check_call([sys.executable, str(profile_py), *profile_args])
else:
    print('Skipping profiling. Set RUN_PROFILE=True to profile chunk/cap choices.')


Optional but recommended: prebuild the microstructure Parquet cache before training. This is the slow CSV decompression step; later training epochs reuse the cache.

In [ ]:
import subprocess
import sys

RUN_PREPROCESS = False  # Set True to build/refresh the microstructure cache before training.
PREPROCESS_PROCESSES = int(manifest.get('default_preprocess_processes', 8))
POLARS_THREADS_PER_PROCESS = int(manifest.get('default_polars_threads_per_process', 2))
REBUILD_PREPROCESS_CACHE = False

preprocess_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'preprocess_event_chunks.py'
preprocess_args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--start-date', manifest['train_start_date'],
    '--end-date', manifest['test_end_date'],
    '--tickers', manifest.get('tickers', 'ALL'),
    '--chunk-ms', '250',
    '--max-quote-events', '96',
    '--max-trade-events', '64',
    '--max-total-events', '128',
    '--processes', str(PREPROCESS_PROCESSES),
    '--polars-threads-per-process', str(POLARS_THREADS_PER_PROCESS),
]
if REBUILD_PREPROCESS_CACHE:
    preprocess_args.append('--rebuild-cache')

if RUN_PREPROCESS:
    print('Running:', ' '.join([str(preprocess_py), *preprocess_args]))
    subprocess.check_call([sys.executable, str(preprocess_py), *preprocess_args])
else:
    print('Skipping preprocessing. Set RUN_PREPROCESS=True to build the cache first.')


In [ ]:
import runpy
import sys

BATCH_SIZE = int(manifest.get('default_batch_size', 4096))
EPOCHS = int(manifest.get('default_epochs', 3))
NUM_WORKERS = int(manifest.get('default_num_workers', 8))
PREFETCH_FACTOR = int(manifest.get('default_prefetch_factor', 4))
MAX_STEPS = 0
COUNT_COVERAGE = False
DRY_RUN = False
REBUILD_CACHE = False

train_py = LOCAL_CODE_ROOT / 'research' / 'inhouse_transformer' / 'v22' / 'train.py'
args = [
    '--flatfiles-root', str(FLATFILES_ROOT),
    '--cache-root', str(CACHE_ROOT),
    '--train-start-date', manifest['train_start_date'],
    '--train-end-date', manifest['train_end_date'],
    '--validation-start-date', manifest['validation_start_date'],
    '--validation-end-date', manifest['validation_end_date'],
    '--test-start-date', manifest['test_start_date'],
    '--test-end-date', manifest['test_end_date'],
    '--device', 'cuda',
    '--output-root', str(OUTPUT_ROOT),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--max-steps', str(MAX_STEPS),
    '--num-workers', str(NUM_WORKERS),
    '--prefetch-factor', str(PREFETCH_FACTOR),
    '--tickers', manifest.get('tickers', 'ALL'),
    '--checkpoint-policy', 'last_only',
    '--wandb-entity', manifest['wandb_entity'],
    '--wandb-project', manifest['wandb_project'],
    '--wandb-run-name', manifest['wandb_run_name'],
    '--output-name', manifest['wandb_run_name'],
]
if REBUILD_CACHE:
    args.append('--rebuild-cache')
if COUNT_COVERAGE:
    args.append('--count-coverage')
if DRY_RUN:
    args.append('--dry-run')

print('Running:', ' '.join([str(train_py), *args]))
old_argv = sys.argv[:]
try:
    sys.argv = [str(train_py), *args]
    runpy.run_path(str(train_py), run_name='__main__')
finally:
    sys.argv = old_argv
